*0.4 Deep learning basics*

# PyTorch tensors

**The situation.** A re-ranking service scores 200 candidate documents for each query. The first version keeps the embeddings as Python lists of floats and loops. It takes 400 ms per query. The same work as tensors takes under a millisecond — and can move to a GPU with one line when the traffic grows.

**Tensors.** A tensor is an n-dimensional array of numbers with a *shape*, a *dtype* (number format) and a *device* (CPU or GPU). Every model input, weight and output in PyTorch is a tensor. Operations run on whole tensors at once, in C++/CUDA, which is where the speed comes from.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

In [2]:
import time

import torch

torch.manual_seed(0)
query = torch.randn(384)  # one query embedding
candidates = torch.randn(200, 384)  # 200 candidate embeddings, one per row

print(
    "shape:", tuple(candidates.shape), "| dtype:", candidates.dtype, "| device:", candidates.device
)
print("memory:", candidates.numel() * candidates.element_size(), "bytes  (200 × 384 × 4)")

started = time.perf_counter()
scores_loop = []
for row in candidates.tolist():  # the list-of-floats version
    total = 0.0
    for a, b in zip(row, query.tolist()):
        total += a * b
    scores_loop.append(total)
loop_ms = (time.perf_counter() - started) * 1000

candidates @ query  # warm-up: the first call pays a one-time setup cost
started = time.perf_counter()
scores = candidates @ query  # the tensor version: one matrix-vector product
tensor_ms = (time.perf_counter() - started) * 1000
print(
    
        f"python loop {loop_ms:.1f} ms | tensor {tensor_ms:.3f} ms | same numbers: "
        f"{torch.allclose(torch.tensor(scores_loop), scores, atol=1e-3)}"
    
)
assert tensor_ms < loop_ms

shape: (200, 384) | dtype: torch.float32 | device: cpu
memory: 307200 bytes  (200 × 384 × 4)
python loop 6.7 ms | tensor 0.032 ms | same numbers: True


**Reading the output.** Same 200 scores, from a loop and from one `@`; the tensor version is orders of magnitude faster. Shape, dtype and device are the three things you will check in every bug.

**The three things that go wrong: shape, dtype, device.** Reshape, change precision to halve memory, and move to the GPU if there is one.

In [3]:
batch = candidates.view(4, 50, 384)  # same numbers, seen as 4 batches of 50
print("reshaped:", tuple(batch.shape), "| batch scores shape:", tuple((batch @ query).shape))

half = candidates.to(torch.float16)  # half precision: half the memory, what GPUs run models in
print(
    "float16 memory:",
    half.numel() * half.element_size(),
    "bytes | largest rounding error:",
    f"{(half.float() - candidates).abs().max():.4f}",
)

device = (
    "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
)
on_device = candidates.to(device)
print("moved to:", on_device.device)
try:
    candidates @ query.to(device) if device != "cpu" else None
    if device != "cpu":
        raise AssertionError("mixing devices should fail")
except RuntimeError as error:
    print(
        "CPU tensor @ GPU tensor →",
        type(error).__name__,
        "(tensors in one operation must share a device)",
    )
assert tuple(batch.shape) == (4, 50, 384)

reshaped: (4, 50, 384) | batch scores shape: (4, 50)
float16 memory: 153600 bytes | largest rounding error: 0.0016
moved to: mps:0
CPU tensor @ GPU tensor → RuntimeError (tensors in one operation must share a device)


**The rule to remember.** Everything is a tensor; every bug is shape, dtype or device. Print all three before anything else.

```
candidates  shape (200, 384)  dtype float32  device cpu
   .view(4, 50, 384)          → reshape, no copy
   .to(torch.float16)         → half memory, tiny rounding
   .to("cuda")                → GPU; every tensor in an operation must be there
```

| Use it when | Don't when | Instead use |
|---|---|---|
| any numeric work that touches a model, a GPU, or gradients | plain numeric analysis with no model | NumPy — and `torch.from_numpy` when you need to cross over |

**Watch out**
- `.view` needs contiguous memory; `.reshape` copies when it must. When `.view` errors, use `.reshape`.
- float16 overflows above 65,504; bfloat16 does not and is what training uses on modern GPUs.
- `.tolist()` / `.item()` copy from GPU to CPU and stall the pipeline; call them once at the end, not inside loops.